# Implement 2D Positional Embeddings

**Difficulty**: 🟡 Medium

**Companies**: Anthropic, DeepMind, Midjourney, Runway

---

### Problem Statement

Vision Transformers (ViT) divide an image into a grid of patches and process them as a sequence. Unlike text, these patches have **two-dimensional spatial structure** (row and column). Standard 1D sinusoidal positional embeddings lose this 2D spatial information.

Your task is to implement **2D sinusoidal positional embeddings** that encode both row and column position using sine/cosine functions. The embedding dimension is split in half: the first half encodes row position and the second half encodes column position, each using the standard sinusoidal formula.

---

### Requirements

1. **`create_2d_sinusoidal_embeddings(height, width, d_model)`** — Returns a tensor of shape `(height * width, d_model)` containing the 2D positional embeddings.
2. **Dimension split** — First `d_model // 2` dimensions encode row position; last `d_model // 2` dimensions encode column position.
3. **Sinusoidal formula** — For each half, use alternating sine and cosine: `PE(pos, 2i) = sin(pos / 10000^(2i/d_half))`, `PE(pos, 2i+1) = cos(pos / 10000^(2i/d_half))`.

---

### Constraints

- ✅ Must work for any `height`, `width`, and even `d_model`.
- ✅ Embeddings at neighboring grid positions should be more similar (higher cosine similarity) than distant positions.
- ✅ Row embeddings and column embeddings should be independent.
- ❌ Do **not** use `nn.Embedding` or learned embeddings.

---

<details>
  <summary>💡 Hint</summary>

  - Compute the frequency divisors: `div_term = torch.exp(torch.arange(0, d_half, 2) * -(math.log(10000.0) / d_half))`.
  - Create position vectors: `row_pos = torch.arange(height).unsqueeze(1)` and similarly for columns.
  - Fill even indices with sine, odd indices with cosine for each half.
  - Concatenate the row and column embeddings, then reshape to `(height * width, d_model)`.

</details>

---

In [ ]:
import math
import torch
import torch.nn.functional as F

In [ ]:
# Configuration
height = 8    # number of patch rows
width = 8     # number of patch columns
d_model = 64  # embedding dimension (must be even)

print(f"Grid: {height}x{width} = {height * width} patches")
print(f"Embedding dimension: {d_model}")
print(f"Each half (row/col): {d_model // 2}")

In [ ]:
def create_2d_sinusoidal_embeddings(height: int, width: int, d_model: int) -> torch.Tensor:
    """
    Create 2D sinusoidal positional embeddings.

    Args:
        height:  number of rows in the patch grid
        width:   number of columns in the patch grid
        d_model: total embedding dimension (must be even)

    Returns:
        Tensor of shape (height * width, d_model)
    """
    assert d_model % 2 == 0, "d_model must be even"
    d_half = d_model // 2

    # TODO: Step 1 - Compute the frequency divisor term for d_half dimensions
    #       div_term = exp(arange(0, d_half, 2) * -(log(10000) / d_half))
    #       Shape: (d_half // 2,)

    # TODO: Step 2 - Create row positional embeddings
    #       row_pos = arange(height).unsqueeze(1)  -> shape (height, 1)
    #       pe_row = zeros(height, d_half)
    #       pe_row[:, 0::2] = sin(row_pos * div_term)
    #       pe_row[:, 1::2] = cos(row_pos * div_term)

    # TODO: Step 3 - Create column positional embeddings (same pattern)
    #       col_pos = arange(width).unsqueeze(1)
    #       pe_col = zeros(width, d_half)
    #       pe_col[:, 0::2] = sin(col_pos * div_term)
    #       pe_col[:, 1::2] = cos(col_pos * div_term)

    # TODO: Step 4 - Combine: for each (row, col) position, concatenate pe_row[row] and pe_col[col]
    #       Use broadcasting: pe_row.unsqueeze(1) -> (height, 1, d_half)
    #                         pe_col.unsqueeze(0) -> (1, width, d_half)
    #       Expand and concatenate along last dim -> (height, width, d_model)
    #       Reshape to (height * width, d_model)

    ...


# Quick test
pe = create_2d_sinusoidal_embeddings(height, width, d_model)
print(f"Positional embeddings shape: {pe.shape}")
assert pe.shape == (height * width, d_model), f"Expected ({height * width}, {d_model}), got {pe.shape}"

In [ ]:
# Validation
pe = create_2d_sinusoidal_embeddings(height, width, d_model)

# Helper: convert (row, col) to flat index
def idx(r, c):
    return r * width + c

# Validation 1: Neighboring positions should be more similar than distant ones
print("=== Neighbor vs Distant Similarity ===")
center = pe[idx(4, 4)]  # center patch
neighbor = pe[idx(4, 5)]  # adjacent patch
distant = pe[idx(0, 0)]  # far corner

sim_neighbor = F.cosine_similarity(center.unsqueeze(0), neighbor.unsqueeze(0)).item()
sim_distant = F.cosine_similarity(center.unsqueeze(0), distant.unsqueeze(0)).item()
print(f"Similarity to neighbor (4,5): {sim_neighbor:.4f}")
print(f"Similarity to distant  (0,0): {sim_distant:.4f}")
assert sim_neighbor > sim_distant, "Neighbor should be more similar than distant position!"
print("PASSED\n")

# Validation 2: Same row positions should share row embeddings
print("=== Row Independence ===")
row_half_a = pe[idx(3, 0), :d_model // 2]
row_half_b = pe[idx(3, 5), :d_model // 2]
assert torch.allclose(row_half_a, row_half_b), "Same row should have identical row embeddings!"
print("Patches (3,0) and (3,5) share row embeddings: PASSED\n")

# Validation 3: Same column positions should share column embeddings
print("=== Column Independence ===")
col_half_a = pe[idx(0, 4), d_model // 2:]
col_half_b = pe[idx(6, 4), d_model // 2:]
assert torch.allclose(col_half_a, col_half_b), "Same column should have identical column embeddings!"
print("Patches (0,4) and (6,4) share column embeddings: PASSED\n")

# Validation 4: Visualize similarity matrix
print("=== Similarity Matrix (first 16 positions) ===")
pe_subset = pe[:16]
sim_matrix = F.cosine_similarity(pe_subset.unsqueeze(0), pe_subset.unsqueeze(1), dim=-1)
print("Similarity matrix shape:", sim_matrix.shape)
print("Diagonal (self-similarity) mean:", sim_matrix.diag().mean().item())
assert torch.allclose(sim_matrix.diag(), torch.ones(16)), "Self-similarity should be 1.0!"
print("PASSED\n")

# Validation 5: Different grid sizes should work
print("=== Different Grid Sizes ===")
for h, w, d in [(4, 6, 32), (16, 16, 128), (3, 7, 64)]:
    pe_test = create_2d_sinusoidal_embeddings(h, w, d)
    assert pe_test.shape == (h * w, d), f"Shape mismatch for ({h}, {w}, {d})"
    print(f"  ({h}x{w}, d={d}): shape {pe_test.shape} OK")
print("PASSED\n")

print("All tests passed!")